# Notebook 07 — Session Quality & Engagement Flags

| Audience | What they get |
|---|---|
| **User / Patient** | How consistently you are practising |
| **Doctor / Clinician** | Abandonment rate, session frequency, time-of-day patterns |
| **App maker** | Retention signals, invalid session rate, engagement health |


In [1]:
DATA_DIR    = '.'
OUT_DIR     = 'outputs'
GROUP_LABEL = 'Group A'
import os, json, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

# ── Make sure output folder exists BEFORE anything tries to write to it ──
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, os.path.dirname(os.path.abspath('hci_utils.py')))
from hci_utils import (load_board_tries, load_bd_sessions, load_piano_sessions,
                        load_piano_movements, compare_groups, save_fig,
                        SHAPE_ORDER, HAND_COLORS, GROUP_COLORS)

df_tries         = load_board_tries(DATA_DIR)
bd_valid, bd_bad = load_bd_sessions(DATA_DIR)
piano_sess, _    = load_piano_sessions(DATA_DIR)
piano_valid      = piano_sess[piano_sess['is_valid']]

print(f'Board: {len(bd_valid)} valid, {len(bd_bad)} abandoned')
print(f'Piano: {len(piano_valid)} valid, {len(piano_sess)-len(piano_valid)} abandoned')


KeyError: 'startedAt'

## Fig 07a — Score distribution

In [ ]:
all_scores = list(bd_valid['sessionScore']) + list(piano_valid['sessionScore'])
if not all_scores:
    print('No valid session scores yet.')
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(all_scores, bins=20, color='#2c4fa0', edgecolor='white', alpha=0.85)
    ax.set_xlabel('Session score')
    ax.set_ylabel('Count')
    ax.set_title(GROUP_LABEL + ' — Session score distribution (all games)', fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'fig07a_session_score_distribution.png', OUT_DIR)
    plt.show()


## Fig 07b — Abandonment rate

In [ ]:
bd_v = len(bd_valid); bd_a = len(bd_bad)
p_v  = len(piano_valid); p_a = len(piano_sess) - p_v

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, title, valid, abandoned in [
    (axes[0], 'Board Drawing', bd_v, bd_a),
    (axes[1], 'Piano',         p_v,  p_a),
]:
    total = valid + abandoned
    if total == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
    else:
        ax.pie([valid, abandoned],
               labels=[f'Valid ({valid})', f'Abandoned ({abandoned})'],
               colors=['#2a6e4f', '#c84b2f'], autopct='%1.0f%%',
               startangle=90, wedgeprops=dict(edgecolor='white', lw=2))
    ax.set_title(title, fontweight='bold')
fig.suptitle(GROUP_LABEL + ' — Session abandonment rate', fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig07b_abandonment_rate.png', OUT_DIR)
plt.show()

print('\n=== APP-MAKER SIGNAL ===')
total_bd = bd_v + bd_a
if total_bd > 0:
    print(f'Board abandonment: {bd_a/total_bd*100:.1f}%')
total_p = p_v + p_a
if total_p > 0:
    print(f'Piano abandonment: {p_a/total_p*100:.1f}%')
print('Above 30% = consider adding an in-game prompt or easier entry-level session.')


## Fig 07c — Time-of-day performance

In [ ]:
# Fix: use pd.Series(...).dt instead of DatetimeIndex.dt
bd_valid = bd_valid.copy()
bd_valid['hour'] = pd.to_datetime(
    pd.Series(bd_valid['time'].values), utc=True, errors='coerce').dt.hour
piano_sess2 = piano_sess.copy()
piano_sess2['hour'] = pd.to_datetime(
    pd.Series(piano_sess2['time'].values), utc=True, errors='coerce').dt.hour

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, df_, score_col, title in [
    (axes[0], bd_valid,   'sessionScore', 'Board Drawing score by hour'),
    (axes[1], piano_sess2[piano_sess2['is_valid']], 'sessionScore', 'Piano score by hour'),
]:
    if df_.empty or df_['hour'].isna().all():
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        continue
    hourly = df_.groupby('hour')[score_col].mean()
    ax.bar(hourly.index, hourly.values, color='#2c4fa0', alpha=0.8, width=0.7)
    ax.set_xlabel('Hour of day (UTC)')
    ax.set_ylabel('Avg score')
    ax.set_title(title, fontweight='bold')
    ax.set_xticks(range(0, 24, 2))
fig.suptitle(GROUP_LABEL + ' — Time-of-day performance', fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig07c_time_of_day.png', OUT_DIR)
plt.show()

print('\n=== CLINICIAN SIGNAL ===')
print('If scores are consistently lower at certain hours, discuss optimal session timing.')
print('\n=== APP-MAKER SIGNAL ===')
print('Hourly engagement pattern = best time to send push notification reminders.')


## Fig 07d — Sessions per day

In [ ]:
# Fix: convert to Series before using .dt
all_time_vals = (list(bd_valid['time'].values) +
                 list(bd_bad['time'].values) +
                 list(piano_sess['time'].values))
all_times_series = pd.to_datetime(pd.Series(all_time_vals), utc=True, errors='coerce').dropna()
days = all_times_series.dt.date.value_counts().sort_index()

if days.empty:
    print('No session timestamps available.')
else:
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.bar(days.index.astype(str), days.values, color='#2c4fa0', alpha=0.85, width=0.5)
    ax.set_xlabel('Date')
    ax.set_ylabel('Number of sessions')
    ax.set_title(GROUP_LABEL + ' — Sessions per day', fontweight='bold')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    save_fig(fig, 'fig07d_sessions_per_day.png', OUT_DIR)
    plt.show()

    print('\n=== APP-MAKER / CLINICIAN SIGNAL ===')
    print(f'Total active days: {len(days)}')
    print(f'Avg sessions per active day: {days.mean():.1f}')
    print(f'Max sessions in a day: {days.max()}')
    print('Gaps > 3 days = dropout risk. Consider a reminder system.')


## Export session quality flags CSV

In [ ]:
flags = []
for _, row in bd_valid.iterrows():
    flags.append({'session_id': row.get('_id',''), 'game': 'board_drawing',
                  'score': row['sessionScore'], 'is_abandoned': False,
                  'hour_of_day': row.get('hour', np.nan)})
for _, row in bd_bad.iterrows():
    h = pd.to_datetime(row.get('time'), utc=True, errors='coerce')
    flags.append({'session_id': row.get('_id',''), 'game': 'board_drawing',
                  'score': 0, 'is_abandoned': True,
                  'hour_of_day': h.hour if not pd.isna(h) else np.nan})
for _, row in piano_sess.iterrows():
    flags.append({'session_id': row['session_id'], 'game': 'piano',
                  'score': row['sessionScore'], 'is_abandoned': not row['is_valid'],
                  'hour_of_day': row.get('hour', np.nan)})

flags_df = pd.DataFrame(flags)
flags_df.to_csv(os.path.join(OUT_DIR, 'session_quality_flags.csv'), index=False)
print('session_quality_flags.csv saved')
print(flags_df.to_string())
